# Generate Figure (The Spite Plot) New as of 2021-07-19

Ok, I basically need to update this notebook to look for information about what symbols and stuff should be shown on the plot. i.e. "show_dp" column of the wd abundance file should be looked to.

This is the same message at the beginning of all jupyter notebooks in this directory. 

If you don't have the below packages, you obviously need to install them for this to work. If it doesn't work still it's extremely likely you have an outdated version of one of the packages. Alternatively, some of the histogram functions actually rely on not being the most recent version because they changed from "normed" to something else from my recollection. Or perhaps it was the other way. I am aware this was poor decision-making, but it works (if you use the right version). ¯\\_(ツ)_/¯

Also pretty much all of these commands are copied and pasted from another Jupyter notebook I made but contained tons of tries at doing this stuff (and unrelated efforts) so that's why a lot of the variables seem unnecessary to use.

In [1]:
from __future__ import print_function

import matplotlib
matplotlib.use('pdf')


import numpy as np
import matplotlib.pyplot as plt
import sys
import os
from astropy.io import fits
from glob import glob
from astropy.time import Time
from astropy import coordinates as coords
from astropy import units as u
from astropy import constants as const
from astropy import convolution as conv
from astropy.table import Table, Column
import scipy.interpolate as scinterp
import time
start = time.time()
print(start)
time_string=str(start).split('.')[0]

import spec_plot_tools as spt
import cal_params as cp
import plot_spec as ps
import bensby_plotting as bp
import abundance_corrections as acorr
import interp_tau as itau
import fix_strings as fs





print(os.getcwd())
print(matplotlib.get_backend())

1634656230.472289
all_fwctb
(116, 4, 27)
(4, 27, 116)
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DA_diff_ov00_diffusion_timescales.csv
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DA_diff_ov10_diffusion_timescales.csv
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DB_diff_ov00_diffusion_timescales.csv
reading in Koester2020 files
retrieving /Users/BenKaiser/Desktop/Goodman_ref_files/diffusion_timescales/Koester2020_DB_diff_ov10_diffusion_timescales.csv
/Users/BenKaiser/Desktop/radial_velocity_calculations
pdf


In [2]:
#figure_output_dir='/Users/BenKaiser/Desktop/GaiaJ1644m0449_paper/Science_versions/Third_Revision/figures'
#figure_output_dir='/Users/BenKaiser/Desktop/GaiaJ1644m0449_paper/ApJ_reformat/figures'
figure_output_dir='/Users/BenKaiser/Desktop'

In [3]:
target_dir= '/Users/BenKaiser/Desktop/radial_velocity_calculations/'
os.chdir(target_dir)

In [4]:
#wd_abund_file='temp_wd_abundances.csv'
#wd_abund_file='20210131_all_wd_abundances.csv'
#wd_abund_file='20210713_all_wd_abundances_bedard_cooling_no_WDs_plot.csv'
wd_abund_file='20210818_all_wd_abundances_bedard_cooling_newer_Blouin.csv'


lodders_abund_file='Lodders2009_solarsystem_abundances.csv'

In [5]:
wd_abund_table=Table.read(wd_abund_file)
wd_abund_table.add_index('name')
print(wd_abund_table['plot_color'])
wd_abund_table=spt.clean_color_string(wd_abund_table,color_header='plot_color')
print(wd_abund_table['plot_color'])

plot_color
----------
 "#ff0000"
 "#8900ff"
 "#00ffc5"
 "#ff33f6"
 "#006e4e"
 "#ff8d00"
 "#1B01F5"
 "#006e4e"
 "#ff8d00"
 "#1B01F5"
plot_color
----------
   #ff0000
   #8900ff
   #00ffc5
   #ff33f6
   #006e4e
   #ff8d00
   #1B01F5
   #006e4e
   #ff8d00
   #1B01F5


In [6]:
lodders_table=Table.read(lodders_abund_file)
lodders_table.add_index('element')

In [7]:
skip_j2356=True
npoints=100

color_dict={
    'GaiaJ1644-0449':'#ff0000',
    'WDJ1644-0449':'#ff0000',
    'SDSSJ1330+6435':'#8900ff',
    'WDJ2356-209':'#00ffc5',
    'SDSSJ1636+1619':'pink',
    'WDJ2317+1830':'orange',
    'WDJ1824+1213':'g',
    'LHS2534':'b'
}
step_dict={
    'GaiaJ1644-0449':5,
    'WDJ1644-0449':5,
    'SDSSJ1330+6435':5,
    'WDJ2356-209':5,
    'SDSSJ1636+1619':5,
    'WDJ2317+1830':5,
    'WDJ1824+1213':5,
    'LHS2534':5
}

t_step=5




fill_dict={
    'GaiaJ1644-0449':'#ffc3c3',#'#ff8282',
    'SDSSJ1330+6435':'#d0a3ff',#'#c689fb',
    'WDJ2356-209':'#ebfffb'#'#d4fb7e'
}

pattern_dict={
    'GaiaJ1644-0449':'#ffc3c3',
    'SDSSJ1330+6435':'#d0a3ff',
    'WDJ2356-209':'#ebfffb'
}


pos_dict={
    'GaiaJ1644-0449':-5,
    'WDJ1644-0449':-5,
    'SDSSJ1330+6435':5,
    'WDJ2356-209':10, #'#b0ff00',
    'SDSSJ1636+1619':15,
    'WDJ2317+1830':-10,
    'WDJ1824+1213':-15,
    'LHS2534':20
}

ssp_pos=2 #offset in addition to object offset for the case of steady-state

#making the points be in the middle of the line
for pos in pos_dict:
    pos_dict[pos]=int(npoints/2.+pos_dict[pos])



line_dict={
    'thin': ':',
    'thick':'-',
    'halo':'-.'
}

wd_marker='*'
#met_marker='D'
#ssp_marker='met_marker'
met_marker='s'
ssp_marker='D'
met_color='#1ca1f2'
#ci_size=14
#wd_size=14
#ci_leg_size=9
#dp_alpha=0.5
#arr_naca=[-0.4,-0.1]
#alpha_range=[0.5,0.2]
#arrow_segs=100
arrow_width=0.03
#arrow_line=4

star_marker='o'
pop_colors=['darkorange','brown','navy','grey'] #thin disk, thick disk, halo, in-between for Bensby plots
#pop_colors=['darkorange','darkorange','darkorange','darkorange'] #for version where we don't distinguish pops

ci_size=10
ci_leg_size=6
wd_size=12
starsize=3
dp_alpha=0.5
arr_naca=[-0.4,-0.1]
alpha_range=[0.5,0.2]
arrow_segs=100
#arrow_width=0.07
arrow_line=4
legend_font=8


#This cell contains a function that hasn't been used in quite awhile, so I changed it to markdown so it doesn't give the impression of being important. (2021-10-11)

def LiHe_to_ALi(row,CaFe, FeH, phase='EP'):
    LiCa= row['li/he']-row['ca/he'] #general number, not relative to solar
    if phase=='EP':
        pass
    elif phase=='SSP':
        LiCa=LiCa+row['tau_ca/tau_li']
    #print('Li/Ca', LiCa)
    CaH=CaFe+FeH #relative to solar
    #print('[Ca/H]', CaH)
    #print('lodders A(Ca)',lodders_table['A_el'][np.where(lodders_table['element']=='Ca')])
    CaH=CaH+lodders_table['A_el'][np.where(lodders_table['element']=='Ca')] #CaH now in A(Ca)
    #print('A(Ca)', CaH)
    LiH=LiCa+CaH #puts LiH immediately on the A(Li) scale
    ALi= LiH
    return ALi

In [8]:
def make_dist(value,sigma, npoints):
    return np.random.normal(loc=value, scale=sigma, size=npoints )




In [9]:
#from Mashonkina et al. 2019 and the other Mashonkina et al. 2019/2020 paper that included thin disk stars as well.
FeH_thick= np.linspace(-1.5,-0.4,npoints)
#FeH_halo=np.linspace(-2.6,-1.2,npoints)
FeH_halo=np.linspace(-3.0,-1.2,npoints) #Arbitrary lower metallicity boundary I added on 2021-07-19
FeH_thin= np.linspace(-0.73, 0.24, npoints)
#from Mashonkina et al. 2019
CaFe_thick=0.24
CaFe_thick_err=0.07
CaFe_halo=0.35
CaFe_halo_err=0.08

#from later Mashonkina et al. 2019/2020 and our by-eye approximation of the trend
def get_CaFe_thin(FeH):
    slope=(0-0.24)/(0+0.7)
    print(slope)
    return slope*FeH

CaFe_thin=get_CaFe_thin(FeH_thin)

-0.34285714285714286


In [10]:
#grisoni_primordial=2.6
grisoni_ALi=2.6
coc_ALi=2.7
primordial_ALi=coc_ALi
def grisoni_crude(FeH):
    """
    return the A(Li) value for the [Fe/H] value using Grisoni et al. 2019 approximated model
    """
    flat_inds= np.where(FeH< -0.71)
    func_inds= np.where(FeH>= -0.71)
    output_array=np.ones(FeH.shape)
    output_array[flat_inds]= primordial_ALi
    output_array[func_inds]= 1.111*(FeH[func_inds]+0.71)+primordial_ALi
    return output_array

In [11]:
ymin=0
ymax=5
def make_plot(save_fig=False):
    plt.errorbar(0, lodders_table.loc['Li']['A_el'], label="CI Chondrites",marker=met_marker, color=met_color, linestyle='None', markersize=ci_leg_size)
    plt.errorbar(0, lodders_table.loc['Li']['A_el'], yerr=lodders_table.loc['Li']['A_el_err'], marker=met_marker,color=met_color, markersize=ci_size, linestyle='none')
    model_FeH= np.linspace(-3, 0.6, 100)
    model_ALi= grisoni_crude(model_FeH)
    #plt.plot(model_FeH, model_ALi, label='Crude Grisoni et al. 2019', linestyle='--', color='k')
    plt.plot(model_FeH, model_ALi, linestyle='--', color='k', label='Galactic Li Enrichment')
    plt.xlabel('[Fe/H]')
    plt.ylabel('A(Li)')
    #plt.legend(loc='best')
    #plt.legend(loc='upper left',fontsize=legend_font)
    #plt.legend(loc='upper left',fontsize=legend_font, frameon=True, framealpha=0, facecolor=None)
    #plt.legend(loc='upper left',fontsize=legend_font, frameon=True, facecolor=None)
    plt.legend(loc='lower left',fontsize=legend_font, frameon=True, framealpha=1)
    plt.xlim(-3, 0.6)
    plt.ylim(ymin,ymax)
    if save_fig:
        
        print(os.getcwd())
        os.chdir(figure_output_dir)
        print(os.getcwd())
        start = time.time()
        print(start)
        time_string=str(start).split('.')[0]
        plt.savefig('figure4_'+time_string+'.pdf')#plt.grid(True)


    plt.show()
    return

In [12]:
def fill_under(FeH, ALi_hi,wd_name,ALi_lo=ymin):
    #plt.fill_between(FeH, ALi_hi, ALi_lo, color=color_dict[wd_name], alpha=spite_alpha)
    plt.fill_between(FeH, ALi_hi, ALi_lo, color=fill_dict[wd_name], hatch='x')
    return

There's no [Fe/H] error for the meteor because it is defined to be 0 as the solar value, so it can't have an uncertainty really...

In [13]:
spt.initiate_science_plot()
n_random=1000
#j1644_LiCa_dist=make_dist(wd_abund_table.loc['GaiaJ1644-0449']['li/ca'],wd_abund_table.loc['GaiaJ1644-0449']['li/ca_err'], n_random)
#j1330_LiCa_dist=make_dist(wd_abund_table.loc['SDSSJ1330+6435']['li/ca'],wd_abund_table.loc['SDSSJ1330+6435']['li/ca_err'], n_random)
#j1330_LiCa_dist=make_dist(j1330_LiCa_DP,wd_abund_table.loc['SDSSJ1330+6435']['li/ca_err'], n_random)

#wd_list=[
#    'SDSSJ1330+6435',
#    'GaiaJ1644-0449',
#    'WDJ2356-209',
#]

wd_list=wd_abund_table['name']

pop_list=[
    'thin',
    'thick',
    'halo'
]
def plot_wd_ALi(row, pop,SSP=True,pos_index=5,add_arrow=False,t_step=t_step):
    print('\n\n')
    wd_name=row['name']
    print(wd_name)
    try:
        pos_index=pos_dict[wd_name]
    except KeyError as error:
        print("KeyError:",error)
        print('Using 0 for the pos_index value')
        pos_index=0
    #wd_lica_ep=wd_abund_table.loc[wd_name]['li/ca']
    wd_lica_ep=row['li/ca']
    #target_row=wd_abund_table.loc[wd_name]
    target_row=row
    #row=target_row
    if ((target_row['show']==1) and (target_row['show_li_evo']==1)):
        print('target_row["plot_color"]',target_row['plot_color'],type(target_row['plot_color']))
        label=fs.fix_display_string(wd_name)
        lica_err=target_row['li/ca_err']
        plot_marker=wd_marker
        markersize=wd_size
        if SSP==True:
            target_lica, target_kca, lica_err, kca_err=acorr.easy_dist_ssp(target_row,['Li','Ca','K'], plot_all=False,tau_rand=True)
            wd_lica_ep=target_lica
            #label=fs.fix_display_string(label)+' SSP'
            label=fs.fix_display_string(label)+' Steady State'
            #plot_marker=met_marker
            plot_marker=ssp_marker
            markersize=ci_size
            pos_index=pos_index+ssp_pos
        else:
            label=fs.fix_display_string(label)+' Photospheric'
        #lica_dist=np.random.normal(wd_lica_ep,lica_err, n_random)
        wd_ALi_err= np.sqrt(lica_err**2+CaFe_thick_err**2+lodders_table.loc['Ca']['A_el_err']**2)
        if ((pop=='thin') and (row['thin_disk']==1)):
            wd_ALi_vals=wd_lica_ep+(CaFe_thin+FeH_thin)+lodders_table.loc['Ca']['A_el']
            #plt.plot(FeH_thin, wd_ALi_vals,  color=color_dict[wd_name], linestyle=line_dict[pop])
            plt.plot(FeH_thin, wd_ALi_vals,  color=target_row['plot_color'], linestyle=line_dict[pop])
            #fill_under(FeH_thin, wd_ALi_vals, wd_name)
        elif ((pop=='thick') and (row['thick_disk']==1)):
            wd_ALi_vals=wd_lica_ep+(CaFe_thick+FeH_thick)+lodders_table.loc['Ca']['A_el']
            #plt.plot(FeH_thick, wd_ALi_vals,  color=color_dict[wd_name], linestyle=line_dict[pop])
            plt.plot(FeH_thick, wd_ALi_vals,  color=target_row['plot_color'], linestyle=line_dict[pop])
            if wd_name=='WDJ2356-209':
                #plt.errorbar(FeH_thick[pos_index], wd_ALi_vals[pos_index],label=fs.fix_display_string(label),yerr=0.2, uplims=True, color=color_dict[wd_name], marker=plot_marker, markersize=markersize,linestyle='None')
                plt.errorbar(FeH_thick[pos_index], wd_ALi_vals[pos_index],label=fs.fix_display_string(label),yerr=0.2, uplims=True, color=target_row['plot_color'], marker=plot_marker, markersize=markersize,linestyle='None')
            else:
                #plt.errorbar(FeH_thick[pos_index],wd_ALi_vals[pos_index], label=fs.fix_display_string(label), color=color_dict[wd_name],marker=plot_marker, markersize=ci_leg_size, linestyle='None')
                #plt.errorbar(FeH_thick[pos_index],wd_ALi_vals[pos_index],yerr=wd_ALi_err, color=color_dict[wd_name],marker=plot_marker, markersize=markersize, linestyle='None')
                plt.errorbar(FeH_thick[pos_index],wd_ALi_vals[pos_index], label=fs.fix_display_string(label), color=target_row['plot_color'],marker=plot_marker, markersize=ci_leg_size, linestyle='None')
                plt.errorbar(FeH_thick[pos_index],wd_ALi_vals[pos_index],yerr=wd_ALi_err, color=target_row['plot_color'],marker=plot_marker, markersize=markersize, linestyle='None')
            if ((add_arrow) and (row['show_dp']==1)):
                tau_time= 10.**itau.extrapolate_tau_x_logg(row['teff'], row['logg'], "Ca",atm_type=row['diff_atm_type'], modeler=acorr.default_modeler, overshoot=acorr.default_overshoot)
                tau_time=tau_time*1e-6 #converted to Myr
                #t_step=5*tau_time #Just dropped this step because it should be defined globally as of 2021-10-08
                t_step=t_step*tau_time
                times=t_step
                #arrow_endy,throwaway=acorr.el1el2_DP_el3el2_ftimes(row['teff'], row["li/ca"], row['na/ca'],times, 'Li', 'Ca', 'Na', logg=row['logg'],atm_type=row['diff_atm_type'])
                arrow_endy,throwaway=acorr.el1el2_DP_el3el2_ftimes(row['teff'], wd_lica_ep, row['na/ca'],times, 'Li', 'Ca', 'Na', logg=row['logg'],atm_type=row['diff_atm_type'])
                arrow_endx=FeH_thick[pos_index]
                arrow_endy=arrow_endy+(CaFe_thick+FeH_thick[pos_index])+lodders_table.loc['Ca']['A_el']
                #print('arrow_endy',arrow_endy,'arrow_endx',arrow_endx)
                #plt.plot(arrow_endx,arrow_endy,marker='o')
                #ypoints=np.linspace(wd_ALi_vals[pos_index],arrow_endy,arrow_segs)
                #xpoints=np.linspace(FeH_thick[pos_index],arrow_endx,arrow_segs)
                #dx=arrow_endx-target_el3el2
                #dy=arrow_endy-target_el1el2
                #dx=arrow_endx-xpoints[-2]
                #dy=arrow_endy-ypoints[-2]
                #def get_segs(points):
                #    return np.vstack([points,np.roll(points,1)]).T[1:]
                #x_segs=get_segs(xpoints)
                #y_segs=get_segs(ypoints)
                #print('x_segs',x_segs)
                #color_array=np.empty_like(ypoints,dtype=str)
                #color_array[:]=color_dict[name]
                #alpha_vals=np.linspace(alpha_range[0],alpha_range[1],arrow_segs)
                #print('alpha_vals',alpha_vals)
                #for x,y,alpha in zip(x_segs, y_segs,alpha_vals):
                    #plt.plot(x,y,color=color_dict[name],alpha=alpha,linewidth=arrow_line)
                 #   plt.plot(x,y,color=color_dict[row['name']],alpha=alpha)
                #plt.arrow(xpoints[-2],ypoints[-2],dx,dy,color=color_dict[row['name']],width=arrow_width, alpha=alpha_range[1],length_includes_head=True )
                #plt.arrow(FeH_thick[pos_index],wd_ALi_vals[pos_index],arrow_endx-FeH_thick[pos_index],arrow_endy-wd_ALi_vals[pos_index],color=color_dict[row['name']],width=arrow_width, alpha=alpha_range[0],length_includes_head=True , linewidth=0)
                plt.arrow(FeH_thick[pos_index],wd_ALi_vals[pos_index],arrow_endx-FeH_thick[pos_index],arrow_endy-wd_ALi_vals[pos_index],color=target_row['plot_color'],width=arrow_width, alpha=alpha_range[0],length_includes_head=True , linewidth=0)

            else:
                pass
            #fill_under(FeH_thick, wd_ALi_vals, wd_name)
        elif ((pop=='halo') and (row['halo']==1)):
            wd_ALi_vals=wd_lica_ep+(CaFe_halo+FeH_halo)+lodders_table.loc['Ca']['A_el']
            #plt.plot(FeH_halo, wd_ALi_vals,  color=color_dict[wd_name], linestyle=line_dict[pop])
            plt.plot(FeH_halo, wd_ALi_vals,  color=target_row['plot_color'], linestyle=line_dict[pop])
            #fill_under(FeH_halo, wd_ALi_vals, wd_name)
        return
    else:
        print('CSV indicates this WD should not be plotted')
        return



#j1644_LiCa_dist=make_dist(wd_abund_table.loc['GaiaJ1644-0449']['li/ca'],wd_abund_table.loc['GaiaJ1644-0449']['li/ca_err'], n_random)
#j1330_LiCa_dist=make_dist(wd_abund_table.loc['SDSSJ1330+6435']['li/ca'],wd_abund_table.loc['SDSSJ1330+6435']['li/ca_err'], n_random)


#CaFe_dist=make_dist(get_CaFe_thin(FeH_thin[-1]), 0.1, n_random)
#ACa_dist=make_dist(lodders_table.loc['Ca']['A_el'], lodders_table.loc['Ca']['A_el_err'],n_random)


#j1330_ALi_vals=wd_abund_table.loc['SDSSJ1330+6435']['li/ca']+(CaFe_thin+FeH_thin)+lodders_table.loc['Ca']['A_el']
#j1644_ALi_vals= wd_abund_table.loc['GaiaJ1644-0449']['li/ca']+(CaFe_thin+FeH_thin)+lodders_table.loc['Ca']['A_el']



#j1644_ALi_dist=j1644_LiCa_dist+CaFe_dist+FeH_thin[-1]+ACa_dist
#j1330_ALi_dist=j1330_LiCa_dist+CaFe_dist+FeH_thin[-1]+ACa_dist
#j1644_ALi_err= np.std(j1644_ALi_dist)
#j1330_ALi_err= np.std(j1330_ALi_dist)
#print(j1644_ALi_err, j1330_ALi_err)

#plt.figure(figsize=(10,10))
#plt.figure(figsize=(4.75,4.75),constrained_layout=True)
plt.figure(figsize=(7.25,7.25),constrained_layout=False)

#spt.start_ApJ_fig(width_cols=2,constrained_layout=True, width_height=[4.75,4.75])


#for wd in wd_list:
#    if ((wd =="WDJ2356-209") and (skip_j2356)):
#        pass
#    else:
#        for pop in pop_list:
#            plot_wd_ALi(wd,pop,SSP=False) #This o ne plots direct photosphere abundances (i.e. the star-shaped points)
#            plot_wd_ALi(wd,pop,add_arrow=True)

for row in wd_abund_table:
    for pop in pop_list:
        plot_wd_ALi(row,pop,SSP=False) #This one plots direct photosphere abundances (i.e. the star-shaped points)
        plot_wd_ALi(row,pop,SSP=True, add_arrow=True)

bp.plot_ALi_FeH(colors=pop_colors, marker=star_marker,markersize=starsize)
plt.errorbar(0, 1.10, yerr=0.1,marker=star_marker, color=met_color, label="Sun",linestyle='None',markersize=starsize )



#plt.plot(FeH_thin,j1330_ALi_vals ,  color=color_dict['SDSSJ1330+6435'], linestyle=line_dict['thin'])
#plt.errorbar(FeH_thin[8], j1330_ALi_vals[8], yerr=j1330_ALi_err, color='r',label=wd_abund_table.loc['SDSSJ1330+6435']['name']+' thin disk EP',marker='o')


#plt.plot(FeH_thin, j1644_ALi_vals,  color=color_dict['GaiaJ1644-0449'], linestyle=line_dict['thin'])
#fill_under(FeH_thin, j1644_ALi_vals, 'GaiaJ1644-0449')
#plt.errorbar(FeH_thin[3], j1644_ALi_vals[3], yerr=j1644_ALi_err, color='b',label=wd_abund_table.loc['GaiaJ1644-0449']['name']+' thin disk EP', marker='o')

###########

#CaFe_dist=make_dist(CaFe_thick, CaFe_thick_err, n_random)
#ACa_dist=make_dist(lodders_table.loc['Ca']['A_el'], lodders_table.loc['Ca']['A_el_err'],n_random)


#j1330_ALi_vals=wd_abund_table.loc['SDSSJ1330+6435']['li/ca']+(CaFe_thick+FeH_thick)+lodders_table.loc['Ca']['A_el']
#j1644_ALi_vals= wd_abund_table.loc['GaiaJ1644-0449']['li/ca']+(CaFe_thick+FeH_thick)+lodders_table.loc['Ca']['A_el']



#j1644_ALi_dist=j1644_LiCa_dist+CaFe_dist+FeH_thick[-1]+ACa_dist
#j1330_ALi_dist=j1330_LiCa_dist+CaFe_dist+FeH_thick[-1]+ACa_dist
#j1644_ALi_err= np.std(j1644_ALi_dist)
#j1330_ALi_err= np.std(j1330_ALi_dist)
#print(j1644_ALi_err, j1330_ALi_err)

#plt.plot(FeH_thick,j1330_ALi_vals ,  color=color_dict['SDSSJ1330+6435'], linestyle=line_dict['thick'])
#plt.errorbar(FeH_thick[5], j1330_ALi_vals[5], label=wd_abund_table.loc['SDSSJ1330+6435']['name']+' thick disk EP',yerr=j1330_ALi_err, color=color_dict['SDSSJ1330+6435'],marker=wd_marker, markersize=wd_size)


#plt.plot(FeH_thick, j1644_ALi_vals,  color=color_dict['GaiaJ1644-0449'], linestyle=line_dict['thick'])
#plt.errorbar(FeH_thick[0], j1644_ALi_vals[0],label=wd_abund_table.loc['GaiaJ1644-0449']['name']+' thick disk EP', yerr=j1644_ALi_err, color=color_dict['GaiaJ1644-0449'], marker=wd_marker, markersize=wd_size)
#fill_under(FeH_thick, j1644_ALi_vals, 'GaiaJ1644-0449')



#j1330_ALi_vals=wd_abund_table.loc['SDSSJ1330+6435']['li/ca']+(CaFe_halo+FeH_halo)+lodders_table.loc['Ca']['A_el']
#j1644_ALi_vals= wd_abund_table.loc['GaiaJ1644-0449']['li/ca']+(CaFe_halo+FeH_halo)+lodders_table.loc['Ca']['A_el']

#plt.plot(FeH_halo,j1330_ALi_vals ,  color=color_dict['SDSSJ1330+6435'], linestyle=line_dict['halo'])
#plt.plot(FeH_halo, j1644_ALi_vals,  color=color_dict['GaiaJ1644-0449'], linestyle=line_dict['halo'])
#fill_under(FeH_halo, j1644_ALi_vals, 'GaiaJ1644-0449')





#plt.plot([0,0,0],[4.0,4.6,4.8],linestyle='None', marker='o',label='Solar relations for scaling')
############


#plt.plot(FeH_thin, LiHe_to_ALi(wd_abund_table.loc['SDSSJ1330+6435'], CaFe_thin, FeH_thin, phase='EP'), label=wd_abund_table.loc['SDSSJ1330+6435']['name']+' thin disk EP')
#plt.plot(FeH_thick, LiHe_to_ALi(wd_abund_table.loc['WDJ2356-209'], CaFe_thick, FeH_thick, phase='EP'),  color=color_dict['WDJ2356-209'], linestyle=line_dict['thick'])
#plt.errorbar(FeH_thick[0], LiHe_to_ALi(wd_abund_table.loc['WDJ2356-209'], CaFe_thick, FeH_thick, phase='EP')[0],label=wd_abund_table.loc['WDJ2356-209']['name']+' thick disk EP',yerr=0.2, uplims=True, color=color_dict['WDJ2356-209'], marker=wd_marker, markersize=wd_size)

#plt.plot(FeH_halo, LiHe_to_ALi(wd_abund_table.loc['WDJ2356-209'], CaFe_halo, FeH_halo, phase='EP'),  color=color_dict['WDJ2356-209'], linestyle=line_dict['halo'])



#plt.axvline(x=0,linestyle=':', color='grey')
#plt.axhline(y=lodders_table.loc['Li']['A_el'],linestyle=':', color='grey')
#plt.title('All assumed to be in Early Phase (or approximately Early Phase at least)')
#plt.grid(True)


make_plot(save_fig=True)




WDJ1644-0449
target_row["plot_color"] #ff0000 <class 'numpy.str_'>



WDJ1644-0449
target_row["plot_color"] #ff0000 <class 'numpy.str_'>
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
3369.3375473981673 7.736830089812765
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
3369.3375473981673 7.736830089812765
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
3369.3375473981673 7.736830089

using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
4383.136316761991 8.17951306936257
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
4383.136316761991 8.17951306936257
tau_Li-tau_Ca 0.5837051456470348 +/- 0.20163804860761023
tau_K-tau_Ca -0.0036650451241077527 +/- 0.19865104985853752
target_ssp Li Ca -2.0844398923319467
target_ssp K Ca -0.2970539967764367
dist ssp Li Ca -2.078883622658017 -2.0783514661004925 0.4172344519059445
dist ssp K Ca -0.2944873432873003 -0.29633495487589223 0.19865104985853752



WDJ2356-209
CSV indicates this WD should not be plotted



WDJ2356-209
CSV indicates this WD should not be plotted



WDJ2356-209
CSV indicates this WD should not be plotted



WDJ2356-209
CSV indicates this WD should not be plotted



WDJ2356-209
CSV indicates this WD should not be plotted



WDJ2356-209
CSV indicates this WD should n

using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
4315.281132647481 8.715045060569288
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
4315.281132647481 8.715045060569288
tau_Li-tau_Ca 0.6442687599485963 +/- 0.20066972664224608
tau_K-tau_Ca 0.0002885345805882521 +/- 0.2007580047717228
target_ssp Li Ca -0.8230575771118505
target_ssp K Ca 0.9195633780757807
dist ssp Li Ca -0.824821505369707 -0.8259049552876131 0.2703650938858519
dist ssp K Ca 0.9211274164627297 0.9197114654194118 0.2007580047717228



WDJ2317+1830
target_row["plot_color"] #ff8d00 <class 'numpy.str_'>



WDJ2317+1830
target_row["plot_color"] #ff8d00 <class 'numpy.str_'>
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot=1.0
using Koester2020 models He overshoot

/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/numpy/core/_asarray.py:136: UserWarning: Warning: converting a masked element to nan.
  return array(a, dtype, copy=False, order=order, subok=True)


/Users/BenKaiser/Desktop/radial_velocity_calculations
/Users/BenKaiser/Desktop
1634656421.6632462


/Library/Frameworks/Python.framework/Versions/3.6/lib/python3.6/site-packages/ipykernel_launcher.py:30: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.


In [14]:
np.log10(2.12e7/7.92e6)

0.42761067933925795

In [15]:
np.log10(9.79e3/2.91e3)

0.5268897028172306

In [16]:
np.log10(3.5e7/9.3e6)

0.5755850957963405